# SpaceRec v2 


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path
from pprint import pprint

import h5py


def find_spacerec_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in (start, *start.parents):
        if (path / "spacerec_v2" / "api.py").is_file():
            return path
    raise FileNotFoundError("Could not find project root containing spacerec_v2/api.py")


ROOT = find_spacerec_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import spacerec_v2.api as spacerec

dataset = 'brca'
sample_id = 'BREAST'

resource_root = ROOT / 'resources' / 'brca'
embedding_dir = resource_root / 'embedding'
mask_dir = resource_root / 'mask'
type_pred_dir = resource_root / 'type_pred'
train_dir = type_pred_dir / 'train_v2'
aggregate_dir = resource_root / 'aggregate_v2'
evaluation_dir = resource_root / 'Evaluation_v2'

xen_dir = resource_root / 'xen'
he_image = xen_dir / 'Xenium_FFPE_Human_Breast_Cancer_Rep1_he_image.ome.tif'
aligned_dir = xen_dir / 'aligned'
syn_vis_dir = resource_root / 'syn_vis'
deconv_dir = resource_root / 'deconv'
rctd_ref_dir = deconv_dir / 'rctd_ref'
gene_list = resource_root / 'gene_list' / 'gene.txt'
brca_merge_json = ROOT / 'spacerec_v2' / 'deconv' / 'brca_type_merge_17to11.json'
deconv_sample_id = 'BREAST_SYNTHETIC_TRANSCRIPT'
synthetic_h5 = syn_vis_dir / 'filtered_synthetic_st_filtered_feature_bc_matrix_transcript.h5'
synthetic_positions_csv = syn_vis_dir / 'synthetic_tissue_positions_transcript_filtered.csv'
sc_ref_h5ad = deconv_dir / 'reference' / 'scRNA_adata_reannotated.h5ad'
anno_dir = resource_root / 'anno'
r_env_value = os.environ.get('SPACEREC_R_ENV')
r_env = Path(r_env_value) if r_env_value else None
anno_r_env_value = os.environ.get('SPACEREC_ANNO_R_ENV')
anno_r_env = Path(anno_r_env_value) if anno_r_env_value else None

step0_force = True
step05_force = True
step1_force = False
rctd_ref_force = True

ROOT


## Step 0: Simulation

Generate aligned Xenium coordinates and transcript-level synthetic Visium inputs from `resources/brca/xen` into `resources/brca/xen/aligned` and `resources/brca/syn_vis`.


In [ ]:
simu_summary = spacerec.simu(
    dataset=dataset,
    xen_dir=xen_dir,
    aligned_dir=aligned_dir,
    output_syn_vis_dir=syn_vis_dir,
    he_image=he_image,
    sc_ref_h5ad=sc_ref_h5ad,
    force=step0_force,
)
pprint(simu_summary)


## Step 0.5: Annotation

Generate the full Xenium annotation output set into `resources/brca/anno`, including annotation tables, figures, Prism tables, embedding metadata, and BPCells matrices.


In [ ]:
anno_summary = spacerec.anno(
    dataset=dataset,
    xen_dir=xen_dir,
    aligned_dir=aligned_dir,
    sc_ref_h5ad=sc_ref_h5ad,
    output_dir=anno_dir,
    r_env_prefix=anno_r_env,
    merge_json=brca_merge_json,
    force=step05_force,
)
pprint(anno_summary)

anno_validation = anno_summary.get('validation', anno_summary)
print('required_outputs:', len(anno_validation['required_outputs']))
print('bpcells_outputs:', len(anno_validation['bpcells_outputs']))
print('n_annotations:', anno_validation['n_annotations'])
print('n_cell_types:', anno_validation['n_cell_types'])


## Step 1: Deconvolution

Run RCTD on transcript-level synthetic Visium inputs and build the merged11 RCTD reference aligned to the synthetic transcript genes.

In [ ]:
deconv_summary = spacerec.deconv(
    dataset=dataset,
    visium_dir=syn_vis_dir,
    sc_ref_h5ad=sc_ref_h5ad,
    output_dir=deconv_dir,
    sample_id=deconv_sample_id,
    output_prefix='RCTD_BREAST_SYNTHETIC_TRANSCRIPT',
    r_env_prefix=r_env,
    reference_filter='brca17',
    merge=True,
    merge_json=brca_merge_json,
    force=step1_force,
)
pprint(deconv_summary)

rctd_ref_summary = spacerec.rctd_ref(
    dataset=dataset,
    sc_ref_h5ad=sc_ref_h5ad,
    synthetic_h5=synthetic_h5,
    positions_csv=synthetic_positions_csv,
    output_dir=rctd_ref_dir,
    gene_list_txt=gene_list,
    r_env_prefix=r_env,
    merge_json=brca_merge_json,
    force=rctd_ref_force,
)
pprint(rctd_ref_summary)

deconv_csv = deconv_dir / 'RCTD_BREAST_SYNTHETIC_TRANSCRIPT_proportions_merged11_spacerec.csv'
mu_ref = rctd_ref_dir / 'rctd_reference_merged11.npy'
deconv_csv, gene_list, mu_ref

## Step 2: Grid Embedding and Mask

Step 2a `spacerec.ge(...)` writes the raw bbox-grid embedding H5 into `resources/brca/embedding`. It does not build the training mask in the notebook flow.

Step 2b `spacerec.mask(...)` writes mask outputs into `resources/brca/mask` and writes the mask-filtered grid embedding H5 back into `resources/brca/embedding`. The default mask mode is H&E-only (`mode='he'`); embedding-assisted correction is used only when `mode='embedding'` is passed explicitly.

v2 embedding defaults to `token1280 + tile2560 = 3840` with no neighbor features. `grid_size` is inferred from `patch_size // 16` by the API.


In [ ]:
grid_embedding_h5 = embedding_dir / 'grid_embedding.h5'

# Step 2 uses the Step 0 synthetic Visium outputs plus the Xenium H&E image.
positions_csv = syn_vis_dir / 'synthetic_tissue_positions_transcript_filtered.csv'
scalefactors_json = syn_vis_dir / 'synthetic_scalefactors_json.json'
thumbnail_png = syn_vis_dir / 'visium_like_spots_on_he_thumbnail_transcript.png'

# Set True only when the raw image/Visium files are available and embedding should be regenerated.
run_grid_embedding = False
patch_size = 480  # grid_size is inferred as patch_size // 16; use 288 for dense18, 480 for dense30.

if run_grid_embedding:
    ge_summary = spacerec.ge(
        dataset=dataset,
        he_image=he_image,
        positions_csv=positions_csv,
        scalefactors_json=scalefactors_json,
        thumbnail_png=thumbnail_png,
        output_dir=embedding_dir,
        output_h5=grid_embedding_h5,
        patch_size=patch_size,
        stride=patch_size // 4,
        concat_local=True,
        concat_nbr=False,
        model='virchow2',
        batch_size=4,
        num_workers=4,
        force=True,
        auto_select_gpu=True,
        enable_progress_bar=True,
        stage_log_stdout=True,
    )
    pprint(ge_summary)

grid_embedding_h5


In [ ]:
mask_h5 = mask_dir / 'he_mask_grid.h5'
training_grid_embedding_h5 = embedding_dir / 'grid_embedding_train_filtered.h5'

# Default mode is H&E-only. Use mode='embedding' only when embedding-assisted correction is explicitly needed.
run_mask = False
mask_mode = 'he'

if run_mask:
    mask_summary = spacerec.mask(
        dataset=dataset,
        grid_embedding_h5=grid_embedding_h5,
        output_dir=mask_dir,
        output_h5=mask_h5,
        training_h5=training_grid_embedding_h5,
        he_image=he_image,
        mode=mask_mode,
        force=True,
    )
    pprint(mask_summary)
else:
    print(f'Mask step is disabled. Expected training embedding: {training_grid_embedding_h5}')

training_grid_embedding_h5


## Step 3.1: `spacerec.stage1(...)`

Train the v2 finetune-mu router/type stage from spot-level cell-type proportions. This step writes the stage 1 checkpoint under `train_dir / 'model/stage1_router'` and exports grid-level and cell-level type predictions.

Training receives `training_grid_embedding_h5`, which has already been filtered by the H&E-derived `is_tissue` mask.

$$p_{ik}=\mathrm{softmax}(g_\psi(x_i))_k$$


In [ ]:
st_input = synthetic_h5  # 10x H5; stage1/stage2 convert it to train_dir/input/*.h5ad.

training_common = dict(
    dataset=dataset,
    st_h5ad=st_input,
    deconv_csv=deconv_csv,
    grid_embedding_h5=training_grid_embedding_h5,
    gene_list=gene_list,
    mu_ref=mu_ref,
    run_dir=train_dir,
    projection_dim=2048,  # ignored when skip_input_projector=True; kept for checkpoint/config compatibility
    skip_input_projector=True,
    type_head_hidden_layers=2,
    factorized_head_hidden_layers=3,
    scale_head_hidden_dim=512,
    delta_head_hidden_dim=2048,
    batch_size=4,
    lr=5e-5,
    alpha=0.1,
    delta_alpha=0.5,
    limit_spots=None,
    auto_select_gpu=True,
    enable_progress_bar=True,
    progress_refresh_rate=1,
    stage_log_stdout=True,
)

stage1_summary = spacerec.stage1(
    **training_common,
    stage1_epochs=70,
    agg=True,
    cell_polygon_csv=aligned_dir / 'he_alignmented_cell_boundaries.csv',
    type_pred_dir=type_pred_dir,
)
stage1_checkpoint = Path(stage1_summary['stage1_checkpoint'])
pprint(stage1_summary)


## Step 3.2: `spacerec.stage2(...)`

Train the expression stage from the Step 3.1 checkpoint. Stage 2 freezes the router/projection path, trains the expression branch against the RCTD reference profile, then exports `grid_predictions.h5`, `grid_pred_expression.h5`, `cell_pred_expression.h5`, `grid_type.csv`, and `grid_expr.h5ad`.

$$\tilde\mu_{ikg}=\frac{\mu^{ref}_{kg}\exp(\delta_{ikg})}{\sum_{g'}\mu^{ref}_{kg'}\exp(\delta_{ikg'})}$$

$$\hat y_{ig}=\sum_k \mathrm{stopgrad}(p_{ik})s_i\tilde\mu_{ikg}$$


In [ ]:
stage2_summary = spacerec.stage2(
    **training_common,
    gene_txt=gene_list,
    stage1_checkpoint=stage1_checkpoint,
    stage2_epochs=70,
    cell_polygon_csv=aligned_dir / 'he_alignmented_cell_boundaries.csv',
    grid_pred_expression_h5=train_dir / 'grid_pred_expression.h5',
    cell_pred_expression_h5=train_dir / 'cell_pred_expression.h5',
)
train_summary = stage2_summary
pprint(stage2_summary)


## Step 4: Aggregate

Aggregate grid-level expression and type predictions to polygon/cell level using the same API style as the original package.

In [ ]:
grid_predictions_h5 = train_dir / 'grid_predictions.h5'
grid_type_csv = train_dir / 'grid_type.csv'
grid_expr_h5ad = train_dir / 'grid_expr.h5ad'
polygon_csv = ROOT / 'data' / 'xen_polygon_fullres.csv'

aggregate_summary = spacerec.agg(
    dataset=dataset,
    grid_expr_h5ad=grid_expr_h5ad,
    grid_type_csv=grid_type_csv,
    polygon_csv=polygon_csv,
    output_dir=aggregate_dir,
    grid_predictions_h5=grid_predictions_h5,
    target_name='xen',
)
pprint(aggregate_summary)

## Step 5: Evaluation

Render the same quick grid and Xenium sanity checks as the original notebook.

In [ ]:
window = (0, 0, 2000, 2000) # Not right! Should select right windows

type_summary = spacerec.plottype(
    dataset=dataset,
    grid_type_csv=grid_type_csv,
    window=window,
    true_xen_type_csv=ROOT / 'data' / 'true_xen_type.csv',
    type_merge_json=ROOT / 'data' / 'true_xen_type_merge.json',
    output_dir=evaluation_dir,
)

expr_summary = spacerec.plotexpr(
    dataset=dataset,
    grid_expr_h5ad=grid_expr_h5ad,
    window=window,
    true_xen_expr_h5=ROOT / 'data' / 'true_xen_expr.h5',
    true_xen_type_csv=ROOT / 'data' / 'true_xen_type.csv',
    output_dir=evaluation_dir,
    gene='EPCAM',
)

pprint(type_summary)
pprint(expr_summary)